# Model Prototyping - CWRU Fault Classification (Blackwell Optimized)

This notebook focuses on training and optimizing machine learning models to detect bearing faults. 

### Hardware Support Note:
This environment is optimized for the **NVIDIA GeForce RTX 5070 (Blackwell)** using **PyTorch Nightly (cu128/cu126)** to support **sm_120** kernels.

### Models Included:
1. **MLP (Multi-Layer Perceptron)**: High-speed baseline using 27 statistical/wavelet features.
2. **1D CNN**: Deep feature extraction from vibration sequences.

### Target Output:
- Trained models in `../data/models/` (.pth and .onnx).

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
from tqdm.notebook import tqdm

# Fix for OpenMP runtime conflict on Windows
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"Compute Capability: {torch.cuda.get_device_capability(0)}")
    print(f"PyTorch Build: {torch.__version__}")

PROCESSED_PATH = "../data/processed/cwru_features.parquet"
MODEL_SAVE_DIR = "../data/models"
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

## 1. Data Preparation
We load the 27 statistical and wavelet energy features extracted previously.

In [ ]:
df = pd.read_parquet(PROCESSED_PATH)

# Features are everything except metadata columns
meta_cols = ['fault_type', 'fault_diameter', 'load', 'rpm', 'label']
X = df.drop(columns=meta_cols).values
y = df['label'].values

# Split: 70% Train, 15% Val, 15% Test
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

# Normalize
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print(f"Train size: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

## 2. Model Architecture: MLP
Improved architecture with **Batch Normalization** and **Dropout** for better generalization.

In [ ]:
class FaultMLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(FaultMLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )
    
    def forward(self, x):
        return self.net(x)

num_features = X_train.shape[1]
num_classes = len(np.unique(y))
mlp_model = FaultMLP(num_features, num_classes).to(DEVICE)
print(mlp_model)

## 3. Training Process

In [ ]:
def train_model(model, train_loader, val_loader, epochs=30, lr=0.001):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}', leave=False)
        for batch_X, batch_y in pbar:
            batch_X, batch_y = batch_X.to(DEVICE), batch_y.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * batch_X.size(0)
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        # Validation
        model.eval()
        val_loss, correct = 0.0, 0
        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                batch_X, batch_y = batch_X.to(DEVICE), batch_y.to(DEVICE)
                outputs = model(batch_X)
                val_loss += criterion(outputs, batch_y).item() * batch_X.size(0)
                _, preds = torch.max(outputs, 1)
                correct += torch.sum(preds == batch_y).item()
        
        train_loss /= len(train_loader.dataset)
        val_loss /= len(val_loader.dataset)
        val_acc = correct / len(val_loader.dataset)
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        print(f'Epoch {epoch+1}/{epochs} | Loss: {train_loss:.4f} | Val Acc: {val_acc:.2%}')
            
    return history

train_loader = DataLoader(TensorDataset(torch.FloatTensor(X_train), torch.LongTensor(y_train)), batch_size=64, shuffle=True)
val_loader = DataLoader(TensorDataset(torch.FloatTensor(X_val), torch.LongTensor(y_val)), batch_size=64)

history = train_model(mlp_model, train_loader, val_loader)


## 4. Evaluation & Export

In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Train')
plt.plot(history['val_loss'], label='Val')
plt.title('Loss History')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history['val_acc'])
plt.title('Val Accuracy')
plt.show()

# Export to ONNX
print("Exporting ONNX model...")
mlp_model.eval()
dummy_input = torch.randn(1, num_features).to(DEVICE)
onnx_path = os.path.join(MODEL_SAVE_DIR, "fault_classifier.onnx")
torch.onnx.export(mlp_model, dummy_input, onnx_path, input_names=['inputs'], output_names=['outputs'])
torch.save(mlp_model.state_dict(), os.path.join(MODEL_SAVE_DIR, "best_fault_mlp.pth"))
print(f"Saved to {onnx_path}")